# ARGUS Phase 5 - notebook 01: SFT of M1 (Qwen3.5-4B) on Colab
Runtime > Change runtime type > **T4 GPU**. Run the cells top to bottom. Set `MODE` in the settings cell: `overfit` first (Step 16 test), then `train`.

In [ ]:
%%capture
# This is the install cell of Unsloth's own Qwen3.5-4B notebook (copied Oct 2026). If Unsloth has changed it, replace this cell
# by the first code cell of the current notebook at https://github.com/unslothai/notebooks (nb/Qwen3_5_(4B)_Vision.ipynb).
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"


In [ ]:
# ---------- settings you may change ----------
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR  = "/content/drive/MyDrive/argus/data"      # the folder where you uploaded ml/train_m1/data/*
OUT_DIR   = "/content/drive/MyDrive/argus/out"       # adapters and predictions are saved here
BASE_MODEL = "unsloth/Qwen3.5-4B"
import os, json, random, hashlib
os.makedirs(OUT_DIR, exist_ok=True)

def read_jsonl(path):
    with open(path, encoding="utf-8") as h:
        return [json.loads(l) for l in h if l.strip()]
MODE = "overfit"      # "overfit" = 50-example test, "train" = real run, "ablation" = train without the encryption family
EPOCHS = 3            # used when MODE is train or ablation
MAX_LEN = 1024        # raised automatically if your longest example needs more


In [ ]:
import torch
from unsloth import FastLanguageModel

def load_model(name, max_len):
    try:
        m, t = FastLanguageModel.from_pretrained(name, max_seq_length=max_len, load_in_4bit=False, use_gradient_checkpointing="unsloth")
    except Exception as e:   # Qwen3.5 is a vision-language model; fall back to the loader Unsloth's own notebook uses
        print("FastLanguageModel failed (%s); using FastVisionModel" % str(e)[:150])
        from unsloth import FastVisionModel
        m, t = FastVisionModel.from_pretrained(name, load_in_4bit=False, use_gradient_checkpointing="unsloth")
    return m, t

def add_lora(m):
    return FastLanguageModel.get_peft_model(m, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], use_gradient_checkpointing="unsloth")

def text_tok(t):
    return getattr(t, "tokenizer", t)   # a processor wraps the real tokenizer

def prompt_text(tok, row):
    msgs = [{"role":"system","content":row["system"]},{"role":"user","content":row["user"]}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def encode(tok, row, max_len):
    p = tok(prompt_text(tok, row), add_special_tokens=False)["input_ids"]
    c = tok(row["assistant"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
    ids = (p + c)[:max_len]
    labels = ([-100] * len(p) + c)[:max_len]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

@torch.no_grad()
def generate_greedy(model, tok, rows, max_new=300, batch=16, sample=None):
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(rows), batch):
        chunk = rows[i:i+batch]
        enc = tok([prompt_text(tok, r) for r in chunk], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        kw = dict(max_new_tokens=max_new, use_cache=True)
        if sample: kw.update(do_sample=True, temperature=sample["temperature"], top_p=0.95, num_return_sequences=sample["n"])
        else: kw.update(do_sample=False)
        gen = model.generate(**enc, **kw)
        new = gen[:, enc["input_ids"].shape[1]:]
        texts = tok.batch_decode(new, skip_special_tokens=True)
        n = sample["n"] if sample else 1
        for j in range(len(chunk)):
            outs.append([t.strip() for t in texts[j*n:(j+1)*n]])
    return outs


In [ ]:
files = {"overfit": "overfit50.jsonl", "train": "sft_train.jsonl", "ablation": "sft_train_ablation.jsonl"}
train_rows = read_jsonl(f"{DATA_DIR}/{files[MODE]}")
# ---- leakage guard (Step 14): no training example may share a full hash with val / B1 / B2 ----
held = set(open(f"{DATA_DIR}/heldout_hashes.txt").read().split())
bad = [r["id"] for r in train_rows if r["full_hash"] in held]
assert not bad, f"LEAKAGE: {len(bad)} training examples are in the held-out hash list, e.g. {bad[:3]}"
print("guard passed;", len(train_rows), "training rows")
model, tokenizer = load_model(BASE_MODEL, MAX_LEN)
tok = text_tok(tokenizer)
assert "<|im_end|>" in tok.get_vocab(), "unexpected chat format: <|im_end|> missing"
lens = sorted(len(tok(prompt_text(tok, r), add_special_tokens=False)["input_ids"]) + len(tok(r["assistant"], add_special_tokens=False)["input_ids"]) + 2 for r in train_rows)
print("token lengths p50/p99/max:", lens[len(lens)//2], lens[int(len(lens)*0.99)], lens[-1])
MAX_LEN = max(MAX_LEN, ((lens[-1] + 63) // 64) * 64)
print("MAX_LEN =", MAX_LEN, "   (write these numbers into ml/train_m1/EXPERIMENT_PLAN.md change log)")


In [ ]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig
model = add_lora(model)
ds = Dataset.from_list([encode(tok, r, MAX_LEN) for r in train_rows])
epochs = 25 if MODE == "overfit" else EPOCHS
lr = 2e-4
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
    args=SFTConfig(per_device_train_batch_size=8 if MODE != "overfit" else 5, gradient_accumulation_steps=2, warmup_steps=5,
        num_train_epochs=epochs, learning_rate=lr, logging_steps=5, optim="adamw_8bit", weight_decay=0.001,
        lr_scheduler_type="linear", seed=3407, output_dir=f"{OUT_DIR}/{MODE}", report_to="none",
        save_strategy="epoch" if MODE != "overfit" else "no", save_total_limit=None,
        remove_unused_columns=False, dataset_text_field="", dataset_kwargs={"skip_prepare_dataset": True}, max_length=MAX_LEN))
stats = trainer.train()
print(stats)


In [ ]:
if MODE == "overfit":
    FastLanguageModel.for_inference(model)
    outs = generate_greedy(model, tok, train_rows)
    exact = sum(1 for r, o in zip(train_rows, outs) if o[0] == r["assistant"])
    print(f"OVERFIT TEST: {exact}/{len(train_rows)} training answers reproduced exactly (want at least 45/50 and a final loss near 0)")
else:
    print("Adapters per epoch are in", f"{OUT_DIR}/{MODE}", " (folders named checkpoint-<step>). Next: run the 'checkpoint predictions' cell.")


## Checkpoint predictions on VALIDATION (run after MODE = train)
Writes `val_preds_epoch<N>.jsonl` to the out folder. Download them and score on your laptop with `python -m ml.train_m1.score_preds ...` to choose the epoch.

In [ ]:
import glob, re
ckpts = sorted(glob.glob(f"{OUT_DIR}/{MODE}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
val = read_jsonl(f"{DATA_DIR}/val_distinct.jsonl")
try: del model, trainer
except NameError: pass
import gc; gc.collect(); torch.cuda.empty_cache()
for n, ck in enumerate(ckpts, 1):
    m, t = load_model(ck, 2048)
    FastLanguageModel.for_inference(m)
    tk = text_tok(t)
    outs = generate_greedy(m, tk, val)
    with open(f"{OUT_DIR}/val_preds_epoch{n}.jsonl", "w", encoding="utf-8") as h:
        for r, o in zip(val, outs): h.write(json.dumps({"input_key": r["input_key"], "content": o[0]}) + "\n")
    print("epoch", n, "done:", ck)
    del m, t; gc.collect(); torch.cuda.empty_cache()


## Export the chosen epoch to GGUF (Step 16)
Set `BEST` to the checkpoint folder you chose on validation. The GGUF lands in the out folder; download it to your laptop.

In [ ]:
BEST = ""   # for example f"{OUT_DIR}/train/checkpoint-300"
NAME = "argus_sft"   # use argus_sft_ablation for the ablation model
assert BEST, "set BEST first"
m, t = load_model(BEST, 2048)
m.save_pretrained_gguf(f"{OUT_DIR}/{NAME}_gguf", t, quantization_method="q4_k_m")
print(os.listdir(f"{OUT_DIR}/{NAME}_gguf"))
